# Chapter 06. Streamlit으로 도서 분류·추천 앱 완성하기

앞 Chapter까지 다음 기능을 만들었습니다.

- Chapter 04: **도서 제목 → TF-IDF → Naive Bayes → 예상 분야**
- Chapter 05: **기존 도서 선택 → TF-IDF → Cosine Similarity → 유사 도서 Top 5**

이번 Chapter에서는 새로운 머신러닝 알고리즘을 배우는 것이 아니라, 앞에서 만든 두 기능을 **Streamlit UI에 연결해서 실제로 사용할 수 있는 작은 웹 앱**으로 완성합니다.

전체 흐름은 다음과 같습니다.

**Streamlit 실행 → CSV 로드 → 분류 모델 준비 → 제목 입력 → 분야 예측 → 추천용 TF-IDF 준비 → 도서 선택 → 유사도 Top 5 → app.py 통합 → 브라우저 검증**

이번 Notebook도 이전 Chapter들과 같은 방식으로 작성합니다.

**해야 할 일 이해 → AI에게 질문 → AI 답변 확인 → 주석 코드 실행 → 출력값 직접 확인 → 실제 app.py 작성 → 브라우저 검증**

> 최종 목표는 Notebook의 분석 기능을 **사용자 입력 → 분석 → 결과 출력**의 앱 흐름으로 연결하는 것입니다.

## 학습 목표

이번 Chapter가 끝나면 다음을 할 수 있어야 합니다.

- Streamlit 앱을 실행할 수 있다.
- CSV 데이터를 앱에서 불러올 수 있다.
- `st.cache_data`와 `st.cache_resource`의 차이를 설명할 수 있다.
- 제목 입력창을 Naive Bayes 분류 모델과 연결할 수 있다.
- 도서 선택창을 코사인 유사도 추천 함수와 연결할 수 있다.
- 분류와 추천 기능을 하나의 `app.py`로 통합할 수 있다.
- 브라우저에서 실제 입력으로 결과를 검증할 수 있다.
- 앱이 실행되는 것과 분석 결과가 적절한 것을 구분할 수 있다.
- 분류와 추천 결과의 한계를 설명할 수 있다.

## 실습 1. 현재 Python 환경과 Streamlit 설치 확인

### AI에게 질문

> Windows와 VS Code에서 Python Notebook을 사용하고 있습니다.  
> Streamlit 앱을 만들려고 합니다.
>
> 1. 현재 Notebook이 쓰는 Python 경로와 버전 확인  
> 2. Streamlit 설치 여부 확인  
> 3. pandas와 scikit-learn 버전 확인  
> 4. 설치되지 않았다면 현재 Notebook Python에 설치하는 방법
>
> 을 초보자가 따라할 수 있게 알려 주세요.

### AI 답변

앞에서 Python 3.12와 3.14 환경이 갈렸던 것처럼, 터미널 Python과 Notebook Python이 다를 수 있습니다. 따라서 먼저 현재 커널이 어떤 Python을 쓰는지 확인합니다.

In [ ]:
# 현재 Notebook이 사용하는 Python을 확인합니다.
import sys

print("Python 실행 파일:")
print(sys.executable)

print("\nPython 버전:")
print(sys.version)

# 필요한 패키지 버전을 확인합니다.
import pandas as pd
import sklearn

print("\npandas 버전:", pd.__version__)
print("scikit-learn 버전:", sklearn.__version__)

# Streamlit 설치 여부를 확인합니다.
try:
    import streamlit as st
    print("streamlit 버전:", st.__version__)
except ModuleNotFoundError:
    print("Streamlit이 현재 Notebook Python에 설치되어 있지 않습니다.")
    print("아래 명령을 Notebook 셀에서 실행하면 현재 커널에 설치할 수 있습니다:")
    print(f'"{sys.executable}" -m pip install streamlit')

### 실습 1 결과 확인 및 정리

Streamlit이 설치되어 있지 않다면 **PowerShell의 기본 python이 아니라 현재 Notebook이 사용하는 Python**에 설치하는 것이 안전합니다.

Notebook 셀에서:

```python
import sys
!{sys.executable} -m pip install streamlit
```

처럼 실행하면 현재 커널 환경에 설치됩니다.

터미널에서 설치하고 싶다면 `sys.executable`에 나온 정확한 Python 경로를 사용하면 됩니다.

설치 후에는 Kernel을 재시작한 뒤 다시 import합니다.

## 실습 2. 최소 Streamlit 앱의 구조 이해하기

### AI에게 질문

> Streamlit에서 가장 작은 app.py를 만들고 싶습니다.  
> 제목과 설명 문장만 출력하는 예제를 작성해 주세요.  
> st.title()과 st.write()의 역할도 설명해 주세요.

### AI 답변

`st.title()`은 큰 제목을, `st.write()`는 일반 설명이나 값을 화면에 표시합니다.

In [ ]:
# 아래 문자열은 실제 app.py에 들어갈 가장 작은 Streamlit 예제입니다.
minimal_app_code = '''
import streamlit as st

st.title("교보문고 베스트셀러 텍스트 분석 앱")
st.write("도서 분야 분류와 유사 도서 추천 기능을 실습합니다.")
'''

print(minimal_app_code)

### 실습 2 결과 확인 및 정리

Streamlit 앱은 일반 Python 스크립트처럼 위에서 아래로 실행됩니다.

최소 앱의 역할은 단순합니다.

- `import streamlit as st` → Streamlit 사용
- `st.title()` → 앱의 큰 제목 표시
- `st.write()` → 설명 문장 표시

실제 실행 명령은 터미널에서:

```powershell
python -m streamlit run notebooks/book-text-ml/app.py
```

처럼 사용할 수 있습니다.

브라우저가 열리고 제목과 설명이 보이면 Streamlit 실행 환경은 정상입니다.

## 실습 3. CSV 경로를 안전하게 잡기

수업자료에서는 `app.py`와 `book_bestseller_clean.csv`를 같은 폴더에 두는 구조입니다.

이번 프로젝트의 실제 폴더는 다음과 같습니다.

```text
C:\dev\llm-data-analysis-course
└─ notebooks
   └─ book-text-ml
      ├─ app.py
      └─ book_bestseller_clean.csv
```

다만 Streamlit을 프로젝트 루트에서 실행하면 단순히:

```python
DATA_PATH = "book_bestseller_clean.csv"
```

라고 쓰는 경우 현재 작업 폴더에 따라 파일을 못 찾을 수 있습니다.

그래서 이번 `app.py`에서는 **app.py 파일 자체가 있는 폴더를 기준으로 CSV를 찾도록** 작성합니다.

In [ ]:
# app.py에서 사용할 안전한 경로 방식의 예입니다.
from pathlib import Path

# Notebook에서는 __file__이 없으므로 예시로 현재 프로젝트 경로를 직접 확인합니다.
PROJECT_DATA_PATH = Path(
    "notebooks/book-text-ml/book_bestseller_clean.csv"
)

print("Notebook에서 CSV 존재:", PROJECT_DATA_PATH.exists())
print("Notebook에서 CSV 경로:", PROJECT_DATA_PATH)

### 실습 3 app.py에서 사용할 방식

실제 `app.py` 안에서는 다음과 같이 작성합니다.

```python
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "book_bestseller_clean.csv"
```

이렇게 하면 Streamlit을 어느 폴더에서 실행하더라도 **app.py와 같은 폴더의 CSV**를 찾습니다.

이번 프로젝트에서는 단순 상대경로보다 이 방식이 더 안전합니다.

## 실습 4. 데이터 로딩 함수 만들기

### AI에게 질문

> Streamlit app.py에서 CSV를 읽는 load_data() 함수를 만들고 싶습니다.
>
> 조건:
> - utf-8-sig
> - 필수 컬럼: 상품명, 분야
> - 필수 컬럼이 없으면 ValueError
> - 상품명 결측치는 빈 문자열
> - 분야 결측치는 미분류
> - 문자열 변환
> - 앞뒤 공백 제거
> - 빈 상품명 제거
> - index 재설정
> - st.cache_data 사용
>
> 각 줄에 초보자용 주석을 달아 주세요.

### AI 답변

데이터 로딩 결과는 사용자 입력마다 새로 읽을 필요가 없으므로 `st.cache_data`로 재사용합니다.

In [ ]:
# Notebook에서 같은 로직을 먼저 실행해 확인합니다.
from pathlib import Path

DATA_PATH = Path(
    "notebooks/book-text-ml/book_bestseller_clean.csv"
)

def load_data_for_check():
    # CSV를 읽습니다.
    df = pd.read_csv(
        DATA_PATH,
        encoding="utf-8-sig",
    )

    # 앱이 반드시 필요로 하는 컬럼입니다.
    required_columns = [
        "상품명",
        "분야",
    ]

    # 필수 컬럼 중 없는 컬럼을 찾습니다.
    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    # 필요한 컬럼이 없으면 바로 오류를 알려 줍니다.
    if missing_columns:
        raise ValueError(
            f"필수 컬럼이 없습니다: {missing_columns}"
        )

    # 상품명을 문자열로 정리합니다.
    df["상품명"] = (
        df["상품명"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # 분야를 문자열로 정리합니다.
    df["분야"] = (
        df["분야"]
        .fillna("미분류")
        .astype(str)
        .str.strip()
    )

    # 빈 상품명은 앱에서 사용할 수 없으므로 제외합니다.
    df = (
        df[df["상품명"] != ""]
        .reset_index(drop=True)
    )

    return df

df_app = load_data_for_check()

print("데이터 크기:", df_app.shape)
print("컬럼:", df_app.columns.tolist())
print("빈 상품명 수:", (df_app["상품명"] == "").sum())
print("분야 결측치 수:", df_app["분야"].isna().sum())

df_app[["상품명", "분야"]].head(10)

### 실습 4 결과 확인 및 정리

앱 UI를 만들기 전에 데이터가 정상적으로 준비되는지 먼저 확인합니다.

`load_data()`에서 확인하는 것은 다음과 같습니다.

1. CSV를 읽을 수 있는가?
2. `상품명`, `분야`가 실제로 존재하는가?
3. 상품명에 빈 값이 남아 있지 않은가?
4. 분야 결측치는 `미분류`로 처리되는가?
5. index가 0부터 다시 이어지는가?

Streamlit의 `@st.cache_data`는 이런 **데이터 로딩 결과를 기억해 두고 재사용**할 때 사용합니다.

## 실습 5. st.cache_data와 st.cache_resource 이해하기

Streamlit은 사용자가 버튼을 누르거나 selectbox 값을 바꾸면 Python 스크립트를 다시 실행할 수 있습니다.

매번 CSV를 다시 읽고 모델을 다시 학습하면 불필요하게 느려질 수 있습니다.

이번 Chapter에서는 다음만 기억하면 됩니다.

### st.cache_data

**데이터 결과를 재사용할 때 사용**

예:

- CSV 로딩 결과
- 전처리된 DataFrame

### st.cache_resource

**무거운 객체나 리소스를 재사용할 때 사용**

예:

- TfidfVectorizer
- Naive Bayes 모델
- 추천용 TF-IDF matrix

즉:

```text
load_data()
→ st.cache_data

train_classifier()
build_recommender()
→ st.cache_resource
```

라고 기억하면 됩니다.

## 실습 6. 분류 모델 준비 함수 이해하기

### AI에게 질문

> Streamlit 앱에서는 Chapter 04처럼 성능 평가를 다시 하는 것이 아니라, 평가가 끝난 뒤 시연용 최종 모델을 만들고 싶습니다.
>
> load_data()에서 데이터를 불러온 뒤
> - 미분류 제외
> - 서로 다른 분야가 2개 이상인지 확인
> - TfidfVectorizer fit_transform
> - MultinomialNB 학습
> - vectorizer와 model 반환
> - st.cache_resource 사용
>
> 흐름으로 작성해 주세요.

### AI 답변

Chapter 04에서는 train/test를 나눠 성능을 평가했습니다. 이번 Chapter는 **평가가 끝난 뒤 앱에서 사용할 최종 시연 모델**을 만드는 단계이므로 사용 가능한 라벨 데이터를 이용해 모델을 학습합니다.

In [ ]:
# Notebook에서 앱의 분류 모델 로직을 먼저 확인합니다.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

def train_classifier_for_check(df):
    # 미분류 데이터는 지도학습 정답으로 사용하지 않습니다.
    train_df = (
        df[df["분야"] != "미분류"]
        .copy()
    )

    # 분류하려면 서로 다른 분야가 최소 2개는 있어야 합니다.
    if train_df["분야"].nunique() < 2:
        raise ValueError(
            "서로 다른 분야가 2개 이상 필요합니다."
        )

    # 상품명을 TF-IDF 벡터로 바꿉니다.
    vectorizer = TfidfVectorizer()

    X = vectorizer.fit_transform(
        train_df["상품명"]
    )

    y = train_df["분야"]

    # Naive Bayes 모델을 학습합니다.
    model = MultinomialNB()
    model.fit(X, y)

    return vectorizer, model, train_df

classifier_vectorizer, classifier_model, classifier_train_df = (
    train_classifier_for_check(df_app)
)

print("분류 학습 데이터 수:", len(classifier_train_df))
print("분야 수:", classifier_train_df["분야"].nunique())
print(
    "분류 TF-IDF feature 수:",
    len(classifier_vectorizer.get_feature_names_out())
)
print("모델 클래스 수:", len(classifier_model.classes_))

### 실습 6 결과 확인 및 정리

이번 함수의 목적은 **앱에서 사용할 최종 분류기**를 준비하는 것입니다.

Chapter 04와 차이를 구분해야 합니다.

### Chapter 04

```text
train/test 분리
→ train에 fit
→ test로 평가
```

### Chapter 06

```text
이미 평가가 끝남
→ 사용 가능한 라벨 데이터 전체로
→ 앱 시연용 최종 모델 학습
```

이번 Chapter에서 같은 학습 데이터로 다시 정확도를 계산해 **새로운 성능 평가처럼 보고하지 않습니다.**

성능 평가는 Chapter 04에서 했습니다.

## 실습 7. 새 제목 예측 로직 확인하기

앱 UI에 연결하기 전에 Python에서 새 제목 하나가 정상적으로 예측되는지 확인합니다.

In [ ]:
# 새 도서 제목 예시입니다.
sample_new_title = "처음 배우는 파이썬 데이터 분석"

# 중요: 새 제목에서는 fit_transform이 아니라 transform만 사용합니다.
sample_vector = classifier_vectorizer.transform(
    [sample_new_title]
)

# 학습된 모델로 분야를 예측합니다.
sample_prediction = classifier_model.predict(
    sample_vector
)[0]

print("입력 제목:", sample_new_title)
print("예상 분야:", sample_prediction)
print("새 제목 vector shape:", sample_vector.shape)

In [ ]:
# 새 제목도 학습 때와 같은 feature 수를 사용하는지 확인합니다.
train_feature_count = len(
    classifier_vectorizer.get_feature_names_out()
)

print("학습 feature 수:", train_feature_count)
print("새 제목 vector 열 수:", sample_vector.shape[1])

print(
    "같은 feature 공간인가?:",
    train_feature_count == sample_vector.shape[1]
)

### 실습 7 결과 확인 및 정리

새 제목에서는 절대로 다시 `fit_transform()`하지 않습니다.

올바른 흐름은:

```text
학습
→ fit_transform()

새 제목
→ transform()
→ predict()
```

이미 학습한 단어 사전과 IDF 기준을 그대로 사용해야 같은 좌표계에서 모델이 예측할 수 있습니다.

## 실습 8. Streamlit 제목 입력 UI 이해하기

### AI에게 질문

> Streamlit에서 사용자가 도서 제목을 입력하고 버튼을 누르면 분야를 예측하게 만들고 싶습니다.
>
> 조건:
> - st.header
> - st.text_input
> - placeholder
> - st.button
> - strip()
> - 빈 입력이면 st.warning
> - 정상 입력이면 transform → predict → st.success
>
> 코드를 작성하고 각 UI 요소의 역할을 설명해 주세요.

### AI 답변

입력창과 버튼을 연결하고, 빈 입력은 모델에 전달하지 않도록 먼저 검사합니다.

In [ ]:
# 이 코드는 Notebook에서 실행하는 UI가 아니라 app.py에 들어갈 구조 예시입니다.
classification_ui_code = '''
st.header("1. 도서 분야 예측")

user_title = st.text_input(
    "도서 제목을 입력하세요",
    placeholder="예: 처음 배우는 파이썬 데이터 분석",
)

if st.button("분야 예측"):
    clean_title = user_title.strip()

    if not clean_title:
        st.warning("도서 제목을 입력해 주세요.")
    else:
        title_vector = classifier_vectorizer.transform(
            [clean_title]
        )

        predicted_category = classifier_model.predict(
            title_vector
        )[0]

        st.success(
            f"예상 분야: {predicted_category}"
        )
'''

print(classification_ui_code)

### 실습 8 결과 확인 및 정리

UI 흐름은 다음과 같습니다.

```text
사용자 제목 입력
→ 버튼 클릭
→ strip()
→ 빈 입력인지 확인
→ 기존 vectorizer.transform()
→ model.predict()
→ st.success()
```

`st.text_input()`은 입력창을 만들고, `st.button()`은 사용자가 실제로 분석을 실행하는 시점을 만듭니다.

빈 문자열을 바로 모델에 넣지 않고 안내 메시지를 먼저 보여 주는 것이 사용자 경험 측면에서도 중요합니다.

## 실습 9. 추천용 TF-IDF 준비하기

### AI에게 질문

> Chapter 05 추천 기능을 Streamlit 앱에 연결하려고 합니다.
>
> load_data()의 상품명 전체를 TfidfVectorizer로 fit_transform해서 추천용 title_matrix를 만들고,
> st.cache_resource를 사용하는 build_recommender() 함수를 작성해 주세요.
>
> 분류용 Vectorizer와 추천용 Vectorizer가 왜 별도인지도 설명해 주세요.

### AI 답변

분류용 TF-IDF는 분야 예측 모델의 입력을 만드는 목적이고, 추천용 TF-IDF는 기존 도서 제목끼리 유사도를 비교하는 목적입니다.

In [ ]:
# Notebook에서 추천용 행렬을 먼저 준비해 확인합니다.
def build_recommender_for_check(df):
    vectorizer = TfidfVectorizer()

    title_matrix = vectorizer.fit_transform(
        df["상품명"]
    )

    return vectorizer, title_matrix

reco_vectorizer, title_matrix = (
    build_recommender_for_check(df_app)
)

print("추천 도서 수:", len(df_app))
print("추천 matrix shape:", title_matrix.shape)

print(
    "도서 수 == matrix 행 수:",
    len(df_app) == title_matrix.shape[0]
)

print(
    "추천 feature 수:",
    len(reco_vectorizer.get_feature_names_out())
)

### 실습 9 결과 확인 및 정리

분류와 추천은 둘 다 TF-IDF를 사용하지만 목적이 다릅니다.

### 분류용

```text
상품명
→ TF-IDF
→ Naive Bayes
→ 분야 예측
```

### 추천용

```text
기존 상품명 전체
→ TF-IDF matrix
→ 기준 도서와 전체 도서 유사도
→ Top 5
```

따라서 앱 코드에서도 `train_classifier()`와 `build_recommender()`를 별도 함수로 둡니다.

## 실습 10. 도서 선택 UI 준비하기

### AI에게 질문

> Streamlit selectbox에서 내부적으로는 DataFrame index를 사용하고, 화면에는 사람이 읽기 쉬운 상품명과 저자를 보여 주고 싶습니다.
>
> 저자 컬럼이 있을 때는 "상품명 | 저자", 없으면 상품명만 표시하도록 format_book(index) 함수를 작성해 주세요.

### AI 답변

`selectbox`의 실제 값은 index를 사용하고, `format_func`를 이용해 화면 표시만 사람이 읽기 쉽게 바꿀 수 있습니다.

In [ ]:
# Notebook에서 format_book 로직을 먼저 확인합니다.
def format_book_for_check(index):
    title = df_app.loc[index, "상품명"]

    if "저자" in df_app.columns:
        author = str(
            df_app.loc[index, "저자"]
        )

        return f"{title} | {author}"

    return title

# 앞의 10개가 화면에 어떻게 보일지 확인합니다.
formatted_books = [
    format_book_for_check(i)
    for i in range(min(10, len(df_app)))
]

formatted_books

### 실습 10 결과 확인 및 정리

사용자 화면에는 제목과 저자를 보여 주지만, 내부 계산에서는 **정확한 DataFrame index**를 사용합니다.

이 방식이 좋은 이유는 같은 제목이 여러 개 있어도 index로 정확히 구분할 수 있기 때문입니다.

Streamlit에서는 다음 구조를 사용합니다.

```python
selected_index = st.selectbox(
    "기준 도서를 선택하세요",
    options=df.index.tolist(),
    format_func=format_book,
)
```

## 실습 11. 추천 함수 앱용으로 준비하기

### AI에게 질문

> Chapter 05의 코사인 유사도 추천을 app.py에서 사용할 함수로 만들고 싶습니다.
>
> 조건:
> - df
> - title_matrix
> - selected_index
> - top_n=5
> - cosine_similarity
> - 자기 자신 제외
> - 유사도 높은 순서
> - 상품명/저자/출판사/분야 중 실제 존재하는 컬럼만 표시
> - 유사도 컬럼 추가
>
> 초보자용 주석을 달아 주세요.

### AI 답변

선택한 도서의 벡터와 전체 도서의 유사도를 계산하고 자기 자신은 제외한 뒤 상위 결과를 반환합니다.

In [ ]:
# 코사인 유사도 함수를 불러옵니다.
from sklearn.metrics.pairwise import cosine_similarity

def recommend_books_for_check(
    df,
    title_matrix,
    selected_index,
    top_n=5,
):
    # 선택한 한 권의 벡터를 가져옵니다.
    selected_vector = title_matrix[
        selected_index
    ]

    # 선택 도서와 전체 도서의 유사도를 계산합니다.
    similarities = cosine_similarity(
        selected_vector,
        title_matrix,
    ).ravel()

    # 자기 자신은 추천하지 않도록 가장 낮은 값으로 바꿉니다.
    similarities[selected_index] = -1

    # 유사도가 높은 index부터 top_n개를 선택합니다.
    top_indices = (
        similarities
        .argsort()[::-1][:top_n]
    )

    # 실제 존재하는 컬럼만 결과에 표시합니다.
    display_columns = [
        col
        for col in [
            "상품명",
            "저자",
            "출판사",
            "분야",
        ]
        if col in df.columns
    ]

    # 추천 결과를 가져옵니다.
    result = df.iloc[
        top_indices
    ][display_columns].copy()

    # 유사도 값을 추가합니다.
    result["유사도"] = (
        similarities[top_indices]
        .round(3)
    )

    return result.reset_index(drop=True)

sample_recommendations = recommend_books_for_check(
    df=df_app,
    title_matrix=title_matrix,
    selected_index=0,
    top_n=5,
)

sample_recommendations

In [ ]:
# 추천 결과가 기본 조건을 만족하는지 확인합니다.
print("추천 수:", len(sample_recommendations))
print("5개 이하인가?:", len(sample_recommendations) <= 5)

# 유사도가 내림차순인지 확인합니다.
similarity_values = (
    sample_recommendations["유사도"]
    .to_numpy()
)

is_descending = np.all(
    similarity_values[:-1]
    >= similarity_values[1:]
)

print("유사도 내림차순:", is_descending)

# 선택 도서 제목과 추천 제목이 같은지 확인합니다.
selected_title = df_app.loc[0, "상품명"]

same_title_exists = (
    sample_recommendations["상품명"]
    == selected_title
).any()

print("선택 제목과 같은 제목이 결과에 있는가?:", same_title_exists)

### 실습 11 결과 확인 및 정리

수업자료의 기본 추천 함수는 자기 자신의 index를 `-1`로 만들어 상위 순위에서 제외합니다.

다만 **같은 제목이 다른 index로 중복 등록된 경우**에는 결과에 같은 상품명이 나타날 수 있습니다.

Chapter 05에서는 동일 제목까지 제외하는 확장 함수를 만들었지만, 이번 Chapter에서는 수업자료의 앱 흐름을 우선 그대로 연결합니다.

브라우저에서 실제 추천 결과를 보고 동일 제목이 문제라면 Chapter 05의 확장 로직으로 교체할 수 있습니다.

## 실습 12. 추천 버튼 UI 연결 이해하기

### AI에게 질문

> Streamlit에서 selectbox로 도서를 고른 뒤 "비슷한 도서 5권 추천" 버튼을 누르면 recommend_books()를 실행하고 st.dataframe()으로 보여 주고 싶습니다.
>
> width="stretch", hide_index=True까지 포함해서 작성해 주세요.

### AI 답변

사용자가 고른 `selected_index`를 추천 함수에 전달하고 결과 DataFrame을 화면에 표시합니다.

In [ ]:
# app.py에 들어갈 추천 UI 코드 구조입니다.
recommendation_ui_code = '''
st.header("2. 비슷한 도서 추천")

selected_index = st.selectbox(
    "기준 도서를 선택하세요",
    options=df.index.tolist(),
    format_func=format_book,
)

if st.button("비슷한 도서 5권 추천"):
    recommendations = recommend_books(
        df,
        title_matrix,
        selected_index,
        top_n=5,
    )

    st.dataframe(
        recommendations,
        width="stretch",
        hide_index=True,
    )
'''

print(recommendation_ui_code)

### 실습 12 결과 확인 및 정리

추천 UI 흐름은 다음과 같습니다.

```text
도서 선택
→ selected_index
→ recommend_books()
→ 자기 자신 제외
→ 유사도 높은 순서 Top 5
→ st.dataframe()
```

화면에 결과가 나온다고 바로 끝내지 않고 **선택한 책이 추천 목록에 다시 들어오는지, 5권이 나오는지, 유사도 순서가 맞는지** 직접 확인해야 합니다.

## 실습 13. 최종 app.py의 구조 이해하기

최종 앱은 다음 다섯 역할로 나누어 생각하면 됩니다.

```text
load_data()
→ 데이터 준비

train_classifier()
→ 분야 예측용 Vectorizer + 모델

build_recommender()
→ 추천용 Vectorizer + matrix

recommend_books()
→ Top 5 추천

Streamlit UI
→ 사용자 입력과 결과 출력
```

코드를 한 줄씩 외우는 것보다 **각 함수가 어떤 책임을 갖는지** 구분하는 것이 중요합니다.

## 실습 14. 최종 app.py 코드 전체 확인

아래 코드는 실제로 GitHub에 별도 `app.py` 파일로도 생성합니다.

Notebook에서는 최종 구조를 읽고 이해하는 용도로 확인합니다.

In [ ]:
final_app_code = r'''
from pathlib import Path

import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB


BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "book_bestseller_clean.csv"


st.set_page_config(
    page_title="베스트셀러 텍스트 분석 앱",
    layout="wide",
)


@st.cache_data
def load_data():
    df = pd.read_csv(
        DATA_PATH,
        encoding="utf-8-sig",
    )

    required_columns = [
        "상품명",
        "분야",
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"필수 컬럼이 없습니다: {missing_columns}"
        )

    df["상품명"] = (
        df["상품명"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["분야"] = (
        df["분야"]
        .fillna("미분류")
        .astype(str)
        .str.strip()
    )

    df = (
        df[df["상품명"] != ""]
        .reset_index(drop=True)
    )

    return df


@st.cache_resource
def train_classifier():
    df = load_data()

    train_df = (
        df[df["분야"] != "미분류"]
        .copy()
    )

    if train_df["분야"].nunique() < 2:
        raise ValueError(
            "서로 다른 분야가 2개 이상 필요합니다."
        )

    vectorizer = TfidfVectorizer()

    X = vectorizer.fit_transform(
        train_df["상품명"]
    )

    y = train_df["분야"]

    model = MultinomialNB()
    model.fit(X, y)

    return vectorizer, model


@st.cache_resource
def build_recommender():
    df = load_data()

    vectorizer = TfidfVectorizer()

    title_matrix = vectorizer.fit_transform(
        df["상품명"]
    )

    return vectorizer, title_matrix


def recommend_books(
    df,
    title_matrix,
    selected_index,
    top_n=5,
):
    selected_vector = title_matrix[
        selected_index
    ]

    similarities = cosine_similarity(
        selected_vector,
        title_matrix,
    ).ravel()

    similarities[selected_index] = -1

    top_indices = (
        similarities
        .argsort()[::-1][:top_n]
    )

    display_columns = [
        col
        for col in [
            "상품명",
            "저자",
            "출판사",
            "분야",
        ]
        if col in df.columns
    ]

    result = df.iloc[
        top_indices
    ][display_columns].copy()

    result["유사도"] = (
        similarities[top_indices]
        .round(3)
    )

    return result.reset_index(drop=True)


try:
    df = load_data()

    classifier_vectorizer, classifier_model = (
        train_classifier()
    )

    _, title_matrix = build_recommender()

except (FileNotFoundError, ValueError) as error:
    st.error(str(error))
    st.stop()


st.title(
    "교보문고 베스트셀러 텍스트 분석 앱"
)

st.write(
    "도서 제목을 이용한 분야 예측과 "
    "유사 도서 추천 기능을 실습합니다."
)


st.header("1. 도서 분야 예측")

user_title = st.text_input(
    "도서 제목을 입력하세요",
    placeholder="예: 처음 배우는 파이썬 데이터 분석",
)

if st.button("분야 예측"):
    clean_title = user_title.strip()

    if not clean_title:
        st.warning(
            "도서 제목을 입력해 주세요."
        )
    else:
        title_vector = (
            classifier_vectorizer
            .transform([clean_title])
        )

        predicted_category = (
            classifier_model
            .predict(title_vector)[0]
        )

        st.success(
            f"예상 분야: {predicted_category}"
        )


st.divider()


st.header("2. 비슷한 도서 추천")


def format_book(index):
    title = df.loc[
        index,
        "상품명",
    ]

    if "저자" in df.columns:
        author = str(
            df.loc[index, "저자"]
        )

        return f"{title} | {author}"

    return title


selected_index = st.selectbox(
    "기준 도서를 선택하세요",
    options=df.index.tolist(),
    format_func=format_book,
)

if st.button("비슷한 도서 5권 추천"):
    recommendations = recommend_books(
        df,
        title_matrix,
        selected_index,
        top_n=5,
    )

    st.dataframe(
        recommendations,
        width="stretch",
        hide_index=True,
    )


st.divider()


st.caption(
    "분류 결과는 제목의 텍스트 패턴을 이용한 예측이며 "
    "실제 서점의 공식 분류와 다를 수 있습니다."
)

st.caption(
    "추천 결과는 제목의 TF-IDF 코사인 유사도를 기반으로 하며 "
    "개별 사용자의 취향을 직접 반영하지 않습니다."
)
'''

print(final_app_code)

### 실습 14 결과 확인 및 정리

최종 `app.py`에서 특히 확인할 부분은 다음과 같습니다.

- `DATA_PATH`를 `app.py` 파일 위치 기준으로 잡았습니다.
- CSV 로딩은 `st.cache_data`를 사용합니다.
- 분류 모델과 추천 matrix는 `st.cache_resource`를 사용합니다.
- 새 제목 예측은 `transform()`만 사용합니다.
- 추천은 선택 도서의 index를 이용합니다.
- 자기 자신은 유사도 `-1`로 만들어 제외합니다.
- 앱 시작 시 파일 또는 필수 컬럼 오류가 있으면 사용자에게 메시지를 보여 주고 중단합니다.

즉 Notebook 코드를 그대로 복사한 것이 아니라 **앱에서 반복 실행될 때 안전하게 동작하도록 함수와 캐시로 분리**했습니다.

## 실습 15. app.py 실행 명령 확인

이번 프로젝트의 로컬 경로는:

```text
C:\dev\llm-data-analysis-course
```

입니다.

프로젝트 루트의 PowerShell에서 다음 명령으로 실행합니다.

```powershell
python -m streamlit run notebooks/book-text-ml/app.py
```

만약 PowerShell의 기본 `python`이 다른 버전을 가리킨다면, Notebook에서 확인한 정확한 Python 경로를 사용할 수 있습니다.

예:

```powershell
& "C:\Users\dkrak\AppData\Local\Programs\Python\Python314\python.exe" -m streamlit run notebooks/book-text-ml/app.py
```

앱을 중지하려면 터미널에서:

```text
Ctrl + C
```

를 누릅니다.

## 실습 16. 브라우저에서 분류 기능 검증

분류 기능은 최소 3가지 경우를 직접 확인합니다.

### 1. 빈 제목

입력 없이 **분야 예측** 버튼 클릭

예상:

```text
도서 제목을 입력해 주세요.
```

### 2. 정상 제목

예:

```text
처음 배우는 파이썬 데이터 분석
```

예상:

```text
예상 분야: ...
```

실제 분야명은 현재 학습 데이터와 모델 결과에 따라 달라질 수 있습니다.

### 3. 다른 제목

서로 다른 분야처럼 보이는 제목을 2~3개 입력해 결과가 항상 같은지만 확인하지 말고 직접 비교합니다.

중요:

> 화면에 결과가 나온다는 것과 예측이 적절하다는 것은 다른 문제입니다.

## 실습 17. 브라우저에서 추천 기능 검증

추천 기능은 기준 도서를 2~3권 바꿔 가며 확인합니다.

### 확인할 것

- 자기 자신이 추천 결과에서 제외되는가?
- 최대 5권이 출력되는가?
- 유사도가 높은 순서로 정렬되는가?
- 상품명과 저자/출판사/분야가 정상적으로 보이는가?
- 실제 제목을 읽었을 때 왜 추천되었는지 어느 정도 설명할 수 있는가?
- 유사도가 0인 결과가 포함되어 있지는 않은가?

특히 마지막 항목은 Chapter 05에서 확인한 것처럼 **제목 기반 TF-IDF의 한계**와 연결됩니다.

## 실습 18. 주요 오류 빠르게 확인하기

### 1. CSV를 찾지 못함

대표 오류:

```text
FileNotFoundError
```

확인:

```powershell
dir .\notebooks\book-text-ml\
```

그리고 다음 파일이 있는지 확인합니다.

```text
app.py
book_bestseller_clean.csv
```

이번 `app.py`는 자기 파일 위치를 기준으로 CSV를 찾기 때문에 실행 위치 차이로 인한 오류를 줄였습니다.

### 2. 필수 컬럼 없음

```python
print(df.columns.tolist())
```

으로 실제 컬럼을 확인합니다.

필수 컬럼은:

```text
상품명
분야
```

입니다.

### 3. TF-IDF empty vocabulary

상품명 데이터가 비어 있지 않은지 확인합니다.

```python
print(df["상품명"].head())
print((df["상품명"].str.len() == 0).sum())
```

### 4. Streamlit이 설치되어 있지 않음

```text
ModuleNotFoundError: No module named 'streamlit'
```

현재 앱을 실행할 Python에 설치합니다.

### 5. 예측 또는 추천 결과가 이상함

바로 알고리즘 오류라고 판단하지 않습니다.

```text
입력 확인
→ 실제 데이터 확인
→ index 확인
→ 학습 데이터와 분야 분포 확인
→ TF-IDF feature 확인
→ 모델/추천의 한계 확인
```

순서로 점검합니다.

## 실습 19. 전체 프로젝트 6개 Chapter 연결하기

이번 프로젝트에서 한 일을 순서대로 연결하면 다음과 같습니다.

```text
Chapter 01
데이터 전처리
        ↓
Chapter 02
단어 빈도와 Word Cloud
        ↓
Chapter 03
CountVectorizer와 TF-IDF
        ↓
Chapter 04
Naive Bayes 분야 분류
        ↓
Chapter 05
Cosine Similarity 유사 도서 추천
        ↓
Chapter 06
Streamlit 분류·추천 앱
```

전체 과정은 다음처럼 정리할 수 있습니다.

```text
원본 데이터
→ 전처리
→ 텍스트 탐색
→ 수치화
→ 머신러닝 분류
→ 유사도 추천
→ 사용자 앱
```

즉 이번 Chapter는 앞의 분석 결과를 **사람이 직접 입력하고 선택할 수 있는 형태로 연결하는 마지막 단계**입니다.

## 실습 20. Chapter 06 최종 결과 Markdown

실제 앱을 실행한 뒤 아래 내용을 본인의 실행 결과에 맞게 확인합니다.

### 구현 기능

- CSV 데이터 로딩
- 제목 기반 분야 예측
- 기존 도서 선택
- 제목 TF-IDF 코사인 유사도 추천
- Top 5 결과 표시
- 오류 메시지 처리
- 캐시를 이용한 반복 계산 감소

### 분류 해석

분류 결과는 현재 학습 데이터의 제목 패턴을 바탕으로 한 예측입니다.

```text
예상 분야가 출력됨
≠ 공식 분야가 반드시 그 분야임
≠ 모델이 제목 의미를 완전히 이해함
≠ 예측이 항상 정답임
```

### 추천 해석

추천 결과는 제목 TF-IDF 코사인 유사도 기반입니다.

```text
유사도 높음
≠ 개인 취향과 일치
≠ 더 좋은 책
≠ 구매 가능성이 높음
```

따라서 앱의 결과를 **현재 데이터와 현재 분석 방법이 만든 결과**라는 범위에서 해석합니다.

## 최종 체크리스트

- [ ] Streamlit이 현재 실행 환경에 설치되어 있다.
- [ ] `app.py`가 존재한다.
- [ ] `book_bestseller_clean.csv`가 같은 폴더에 존재한다.
- [ ] CSV가 정상적으로 로드된다.
- [ ] 필수 컬럼 `상품명`, `분야`를 확인했다.
- [ ] `st.cache_data`의 역할을 설명할 수 있다.
- [ ] `st.cache_resource`의 역할을 설명할 수 있다.
- [ ] 도서 제목 입력창이 보인다.
- [ ] 빈 제목 입력을 처리한다.
- [ ] 새 제목은 기존 Vectorizer의 `transform()`을 사용한다.
- [ ] 예상 분야가 화면에 출력된다.
- [ ] 기존 도서를 selectbox에서 선택할 수 있다.
- [ ] 자기 자신을 제외한 유사 도서가 출력된다.
- [ ] 최대 5권이 출력된다.
- [ ] 유사도 순서를 확인했다.
- [ ] 서로 다른 입력으로 분류를 2~3회 검증했다.
- [ ] 서로 다른 기준 도서로 추천을 2~3회 검증했다.
- [ ] 화면이 나온다는 것과 분석 결과가 적절하다는 것을 구분할 수 있다.
- [ ] 분류와 추천 결과의 한계를 설명할 수 있다.

## 이번 Chapter에서 꼭 기억할 5가지

1. **Chapter 06은 새로운 알고리즘보다 기존 분석 기능을 UI에 연결하는 단계입니다.**
2. **새 제목 예측에는 기존 Vectorizer의 transform()을 사용합니다.**
3. **캐시는 반복되는 데이터 로딩과 모델 생성을 줄이는 데 사용합니다.**
4. **추천 결과는 제목 TF-IDF 유사도이며 개인 취향 추천이 아닙니다.**
5. **화면이 정상적으로 보여도 실제 분류와 추천 결과를 직접 검증해야 합니다.**

### Chapter 06 한 문장 정리

**앞에서 만든 TF-IDF 분류와 코사인 유사도 추천 기능을 Streamlit의 입력·선택 UI에 연결하고, 실제 사용자 관점에서 결과를 검증하여 하나의 데이터 분석 앱으로 완성합니다.**